<a href="https://colab.research.google.com/github/Pere91/SAC_Spark/blob/main/SAC_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ensure installation of Java and Spark. Might not be necessary on Google Collab.

In [1]:
!apt-get update -qq
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Get environment variables to set the correct path.

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["SPARK_HOME"], "bin")

Import tools to build the graph and process it with spark.

In [3]:
from pyspark import SparkConf, SparkContext
import networkx as nx

conf = SparkConf().setMaster("local").setAppName("SAC_Spark")
sc = SparkContext.getOrCreate(conf=conf)

Build the directed graph representing the cities and the roads.

In [12]:
G1 = nx.DiGraph()
G1.add_weighted_edges_from([
    ("A", "B", 3.0), ("A", "C", 10.0), ("A", "E", 4.0),
    ("B", "C", 2.0), ("B", "D", 8.0),  ("B", "F", 7.0),
    ("C", "D", 5.0), ("C", "G", 3.0),
    ("D", "H", 6.0),
    ("E", "F", 2.0), ("E", "I", 9.0),
    ("F", "G", 1.0), ("F", "J", 5.0),
    ("G", "H", 2.0), ("G", "K", 4.0),
    ("I", "J", 3.0),
    ("J", "K", 6.0)
])

More examples of graphs.

In [30]:
G2 = nx.DiGraph()
G2.add_weighted_edges_from([
    ("A", "C", 3.0), ("A", "F", 2.0),
    ("C", "D", 4.0), ("C", "F", 2.0), ("C", "E", 1.0),
    ("F", "E", 3.0), ("F", "G", 5.0), ("F", "B", 6.0),
    ("E", "B", 2.0),
    ("D", "B", 1.0),
    ("A", "C", 3.0),
    ("G", "B", 2.0)
])


G3 = nx.DiGraph()
G3.add_weighted_edges_from([
    ("A", "B", 2.0), ("A", "C", 9.0), ("A", "D", 5.0),
    ("B", "E", 4.0),
    ("C", "E", 6.0), ("C", "F", 3.0),
    ("D", "F", 7.0),
    ("E", "G", 2.0), ("E", "H", 8.0),
    ("F", "H", 4.0),
    ("G", "I", 1.0),
    ("H", "I", 6.0)
])


# Select the graph
G = G1

Helper functions.

In [20]:
def get_neighbors(node):
  """
  Provides the neighbor nodes of a given node.

  Args:
      node (tuple): Node to get neighbors from.

  Returns:
      list: List of neighbor nodes.
  """
  return node[1][0]

In [22]:
def mark_visited(node):
  """
  Marks a node as visited.

  Args:
      node (tuple): Node to mark as visited.

  Returns:
      tuple: Node with visited flag,
  """
  return (node[0], (node[1][0], node[1][1], True, node[1][3]))

In [21]:
def generate_path(src, dst):
  """
  Generates the path between two nodes.

  Args:
      src (tuple): Source node.
      dst (tuple): Destination node.

  Returns:
      list: List of nodes in the path, including src and dst.
  """
  return src[1][3] + [dst[0]]

Function that implements Dijkstra's shortest path algorithm, using Spark primitives.

In [23]:
def dijkstra(src):
  """
  Implements Dijkstra's shortest path algorithm, using Spark primitives.

  Args:
      src (str): Source node.

  Returns:
      list: List of nodes in the shortest path from src to all other nodes.
  """

  # Build graph taking node 'src' as starting node
  pyspark_graph = []
  graph_dict = {}

  for node in G.nodes():
    neighbors = [(nbr, G.edges[node, nbr]["weight"]) for nbr in G.successors(node)]

    if node == src:
      weight = 0
      path = [src]
    else:
      weight = float("inf")
      path = []

    pyspark_graph.append((node, (neighbors, weight, False, path)))
    graph_dict[node] = (neighbors, weight, False, path)


  vertices = sc.parallelize(pyspark_graph)

  # Iterate until all nodes have been explored
  while True:

    # Get the unvisited nodes
    unvisited_nodes = vertices.filter(lambda x: not x[1][2])
    if unvisited_nodes.isEmpty():
      break

    # Find the lowest cost node
    next_node = unvisited_nodes.reduce(lambda x, y: x if x[1][1] < y[1][1] else y)

    # Mark node as visited
    next_node = mark_visited(next_node)

    # Get current node neighbors that need updating
    neighbor_list = get_neighbors(next_node)
    neighbor_dict = dict(neighbor_list)
    neighbors = vertices.filter(lambda x: x[0] in [n[0] for n in neighbor_list] and min(neighbor_dict[x[0]] + next_node[1][1], x[1][1]) == neighbor_dict[x[0]] + next_node[1][1])

    # Update costs and paths
    updated_neighbors = neighbors.map(lambda x: (x[0], (x[1][0], neighbor_dict[x[0]] + next_node[1][1], x[1][2], generate_path(next_node, x)))).collect()

    # Update the full graph
    graph_dict[next_node[0]] = next_node[1]
    graph_dict.update(dict(updated_neighbors))
    pyspark_graph = list(graph_dict.items())
    vertices = sc.parallelize(pyspark_graph)

  return vertices.collect()


Call the shortest path algorithm for a certain node.

In [31]:
dijkstra('A')

[('A', ([('B', 3.0), ('C', 10.0), ('E', 4.0)], 0, True, ['A'])),
 ('B', ([('C', 2.0), ('D', 8.0), ('F', 7.0)], 3.0, True, ['A', 'B'])),
 ('C', ([('D', 5.0), ('G', 3.0)], 5.0, True, ['A', 'B', 'C'])),
 ('E', ([('F', 2.0), ('I', 9.0)], 4.0, True, ['A', 'E'])),
 ('D', ([('H', 6.0)], 10.0, True, ['A', 'B', 'C', 'D'])),
 ('F', ([('G', 1.0), ('J', 5.0)], 6.0, True, ['A', 'E', 'F'])),
 ('G', ([('H', 2.0), ('K', 4.0)], 7.0, True, ['A', 'E', 'F', 'G'])),
 ('H', ([], 9.0, True, ['A', 'E', 'F', 'G', 'H'])),
 ('I', ([('J', 3.0)], 13.0, True, ['A', 'E', 'I'])),
 ('J', ([('K', 6.0)], 11.0, True, ['A', 'E', 'F', 'J'])),
 ('K', ([], 11.0, True, ['A', 'E', 'F', 'G', 'K']))]